# OFETNow-ML Generic Prediction Notebook

This notebook is a generic inference workflow for the released OFETNow-ML models.

It supports both JSON and CSV input. JSON is recommended. If a CSV file is used, the notebook will automatically convert it to JSON before prediction.

The workflow uses the released feature-name files directly:

- `models/all_features.txt`
- `models/mobility_selected_features.txt`
- `models/carrier_type_selected_features.txt`

The notebook does **not** require training data and does **not** output benchmark-only columns such as `mobility_label`, `Mobility_Correct`, or `Carrier_Type_Correct`.


## 1. Path Settings

Default paths assume the standard GitHub layout with `models/` and `data/` folders in the repository root. Modify this cell when using custom paths.


In [ ]:
from pathlib import Path

# Recommended default when running from the repository root.
# PROJECT_ROOT = Path('.').resolve()  # Use this if running from repository root.

# If this notebook is placed inside notebooks/, uncomment the next line:
PROJECT_ROOT = Path('..').resolve()

# Optional absolute-path example for local testing:
# PROJECT_ROOT = Path(r'C:\Users\')

MODEL_DIR = PROJECT_ROOT / 'models'
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input can be .json or .csv.
# Default JSON example:
INPUT_PATH = DATA_DIR / 'example_input.json'

# Example CSV input:
# INPUT_PATH = DATA_DIR / 'future_blind.csv'

CSV_TO_JSON_OUTPUT = DATA_DIR / f'{INPUT_PATH.stem}.json'
PREDICTION_OUTPUT_CSV = OUTPUT_DIR / f'{INPUT_PATH.stem}_predictions.csv'
PREDICTION_OUTPUT_JSON = OUTPUT_DIR / f'{INPUT_PATH.stem}_predictions.json'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('MODEL_DIR:', MODEL_DIR)
print('DATA_DIR:', DATA_DIR)
print('INPUT_PATH:', INPUT_PATH)


## 2. Imports


In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import joblib

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import rdFingerprintGenerator

warnings.filterwarnings('ignore')
print('Dependencies loaded.')


## 3. Load Models, Preprocessing Bundle, and Feature Lists


In [ ]:
def read_feature_list(path: Path):
    path = Path(path)
    return [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def require_file(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return path


PREPROCESSING_BUNDLE_PATH = require_file(MODEL_DIR / 'preprocessing_bundle.pkl')
MOBILITY_MODEL_PATH = require_file(MODEL_DIR / 'mobility_classifier.pkl')
CARRIER_MODEL_PATH = require_file(MODEL_DIR / 'carrier_type_classifier.pkl')
ALL_FEATURES_PATH = require_file(MODEL_DIR / 'all_features.txt')
MOBILITY_FEATURES_PATH = require_file(MODEL_DIR / 'mobility_selected_features.txt')
CARRIER_FEATURES_PATH = require_file(MODEL_DIR / 'carrier_type_selected_features.txt')

bundle = joblib.load(PREPROCESSING_BUNDLE_PATH)
mobility_model = joblib.load(MOBILITY_MODEL_PATH)
carrier_model = joblib.load(CARRIER_MODEL_PATH)

all_features = read_feature_list(ALL_FEATURES_PATH)
mobility_selected_features = read_feature_list(MOBILITY_FEATURES_PATH)
carrier_selected_features = read_feature_list(CARRIER_FEATURES_PATH)

objects = bundle.get('objects', bundle) if isinstance(bundle, dict) else {}
onehot_encoder = objects['onehot_encoder']
robust_scaler = objects['robust_scaler']
minmax_scaler = objects['minmax_scaler']
zscore_scaler = objects['zscore_scaler']

print('All features:', len(all_features))
print('Mobility selected features:', len(mobility_selected_features))
print('Carrier-type selected features:', len(carrier_selected_features))
print('Mobility model classes:', getattr(mobility_model, 'classes_', None))
print('Carrier-type model classes:', getattr(carrier_model, 'classes_', None))


## 4. Input Loading: JSON or CSV

CSV input is converted to JSON first. The active prediction input is then loaded into a DataFrame.


In [ ]:
def dataframe_to_json_records(df: pd.DataFrame, json_path: Path):
    records = df.where(pd.notna(df), None).to_dict(orient='records')
    json_path.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding='utf-8')
    return json_path


def load_prediction_input(input_path: Path):
    input_path = Path(input_path)
    if not input_path.exists():
        raise FileNotFoundError(f'Input file not found: {input_path}')

    suffix = input_path.suffix.lower()
    if suffix == '.json':
        with input_path.open('r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = [data]
        df = pd.DataFrame(data)
        active_json_path = input_path
    elif suffix == '.csv':
        try:
            df = pd.read_csv(input_path, encoding='utf-8-sig')
        except UnicodeDecodeError:
            df = pd.read_csv(input_path, encoding='ISO-8859-1')
        active_json_path = OUTPUT_DIR / input_path.with_suffix('.json').name
        dataframe_to_json_records(df, active_json_path)
        print(f'CSV converted to JSON: {active_json_path}')
    else:
        raise ValueError('Input file must be .json or .csv')

    return df, active_json_path


raw_df, ACTIVE_JSON_PATH = load_prediction_input(INPUT_PATH)
print('Active JSON input:', ACTIVE_JSON_PATH)
print('Input shape:', raw_df.shape)
display(raw_df.head())


## 5. Feature Engineering

This section keeps the original trained feature construction order:

`X_numeric + X_categorical + X_morgan + X_descriptor`

The final matrix is converted to a DataFrame using `all_features.txt`, then selected by feature names from the two selected-feature files.


In [ ]:
SMILES_COLUMN = 'SMILES'

ROBUST_FEATURES = ['N_AVE', 'Capacitance', 'Aspect_Ratio', 'Annealing_Temperature']
MINMAX_FEATURES = ['PDI']
CATEGORICAL_FEATURES = ['GATE', 'Dielectric', 'Fabrication_Method', 'Solvent_of_Spin_Coating',
                        'Test_Environment', 'Type', 'S/D']

# The original model uses 2048-bit Morgan fingerprints with radius=8.
MORGAN_RADIUS = 8
MORGAN_N_BITS = 2048

# Fallback descriptor list used only when zscore_scaler does not store feature_names_in_.
SELECTED_DESCRIPTOR_NAMES_FALLBACK = ['SMILES', 'MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'AvgIpc', 'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 'HallKierAlpha', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA5', 'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'SMR_VSA1', 'SMR_VSA10', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7', 'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA10', 'SlogP_VSA12', 'SlogP_VSA2', 'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA6', 'SlogP_VSA8', 'TPSA', 'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 'EState_VSA9', 'VSA_EState1', 'VSA_EState10', 'VSA_EState2', 'VSA_EState3', 'VSA_EState4', 'VSA_EState5', 'VSA_EState6', 'VSA_EState7', 'VSA_EState8', 'FractionCSP3', 'HeavyAtomCount', 'NOCount', 'NumAliphaticHeterocycles', 'NumAliphaticRings', 'NumAmideBonds', 'NumAromaticCarbocycles', 'NumAromaticHeterocycles', 'NumAromaticRings', 'NumHAcceptors', 'NumHeteroatoms', 'NumHeterocycles', 'NumRotatableBonds', 'Phi', 'RingCount', 'MolLogP', 'MolMR', 'fr_Ar_N', 'fr_C_O', 'fr_C_O_noCOO', 'fr_NH0', 'fr_amide', 'fr_aniline', 'fr_aryl_methyl', 'fr_benzene', 'fr_bicyclic', 'fr_ester', 'fr_ether', 'fr_furan', 'fr_halogen', 'fr_imide', 'fr_nitrile', 'fr_para_hydroxylation', 'fr_pyridine', 'fr_sulfide', 'fr_thiophene', 'fr_unbrch_alkane']


def get_descriptor_names_from_scaler(zscore_scaler):
    if hasattr(zscore_scaler, 'feature_names_in_'):
        return ['SMILES'] + list(zscore_scaler.feature_names_in_)
    return SELECTED_DESCRIPTOR_NAMES_FALLBACK


SELECTED_DESCRIPTOR_NAMES = get_descriptor_names_from_scaler(zscore_scaler)
print('Descriptor columns including SMILES:', len(SELECTED_DESCRIPTOR_NAMES))


In [ ]:
def clean_input_dataframe(df: pd.DataFrame):
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('\r', '', regex=False)
            .str.replace('\n', '', regex=False)
            .str.strip()
        )
        df.loc[df[col].isin(['nan', 'None', 'NaN']), col] = ''

    required = [SMILES_COLUMN, 'PDI', 'Aspect_Ratio', 'Capacitance',
                'GATE', 'Dielectric', 'Fabrication_Method', 'Solvent_of_Spin_Coating',
                'Test_Environment', 'Type', 'S/D', 'Annealing_Temperature']
    missing_required = [c for c in required if c not in df.columns]
    if missing_required:
        raise ValueError(f'Missing required input columns: {missing_required}')

    for col in ['Mn_Kda', 'N_AVE', 'PDI', 'Aspect_Ratio', 'Capacitance', 'Annealing_Temperature']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill N_AVE when absent or partially missing.
    if 'Mn_Kda' not in df.columns:
        df['Mn_Kda'] = np.nan

    if 'N_AVE' not in df.columns:
        df['N_AVE'] = np.nan

    missing_nave = df['N_AVE'].isna()
    if missing_nave.any():
        if df['Mn_Kda'].isna().all():
            print(f'Warning: {missing_nave.sum()} entries have no N_AVE and no Mn_Kda. '
                  'N_AVE will remain NaN for these entries.')
        else:
            print(f'Calculating N_AVE for {missing_nave.sum()} entries from Mn_Kda and monomer molecular weight.')
        def monomer_mw(smi):
            mol = Chem.MolFromSmiles(str(smi))
            return Descriptors.MolWt(mol) if mol is not None else np.nan
        monomer_mw_values = df.loc[missing_nave, SMILES_COLUMN].apply(monomer_mw)
        df.loc[missing_nave, 'N_AVE'] = df.loc[missing_nave, 'Mn_Kda'] * 1000 / monomer_mw_values

    return df


def smiles_to_morgan(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.zeros((MORGAN_N_BITS,), dtype=np.uint8)
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_N_BITS)
    fp = generator.GetFingerprint(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.uint8)
    for i in range(fp.GetNumBits()):
        arr[i] = fp.GetBit(i)
    return arr


def generate_normalized_descriptors(df: pd.DataFrame):
    all_descriptor_names = [name for name, _ in Descriptors._descList]
    calculator = MoleculeDescriptors.MolecularDescriptorCalculator(all_descriptor_names)

    rows = []
    for smiles in df[SMILES_COLUMN]:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is not None:
            rows.append(calculator.CalcDescriptors(mol))
        else:
            rows.append([np.nan] * len(all_descriptor_names))

    desc_df = pd.DataFrame(rows, columns=all_descriptor_names)
    desc_df.insert(0, 'SMILES', df[SMILES_COLUMN].values)

    missing_desc = [c for c in SELECTED_DESCRIPTOR_NAMES if c not in desc_df.columns]
    if missing_desc:
        raise ValueError(f'Missing RDKit descriptors, possibly due to RDKit version differences: {missing_desc}')

    selected = desc_df[SELECTED_DESCRIPTOR_NAMES]
    descriptor_numeric = selected.iloc[:, 1:]
    descriptor_numeric = descriptor_numeric.replace([np.inf, -np.inf], np.nan).fillna(0)
    return zscore_scaler.transform(descriptor_numeric)


def build_all_features(df: pd.DataFrame):
    df = clean_input_dataframe(df)

    print('Scaling numeric features...')
    X_robust = robust_scaler.transform(df[ROBUST_FEATURES])
    X_minmax = minmax_scaler.transform(df[MINMAX_FEATURES])
    X_numeric = np.concatenate([X_robust, X_minmax], axis=1)

    print('Encoding categorical features...')
    X_categorical = onehot_encoder.transform(df[CATEGORICAL_FEATURES])
    if hasattr(X_categorical, 'toarray'):
        X_categorical = X_categorical.toarray()

    print('Generating Morgan fingerprints: radius=8, fpSize=2048...')
    X_morgan = np.array(df[SMILES_COLUMN].apply(smiles_to_morgan).tolist())

    print('Generating and scaling RDKit descriptors...')
    X_descriptor = generate_normalized_descriptors(df)

    X_all_array = np.concatenate([X_numeric, X_categorical, X_morgan, X_descriptor], axis=1)
    if X_all_array.shape[1] != len(all_features):
        raise ValueError(
            f'Feature dimension mismatch: generated {X_all_array.shape[1]} columns, '
            f'but all_features.txt contains {len(all_features)} names.'
        )

    X_all = pd.DataFrame(X_all_array, columns=all_features, index=df.index)
    return X_all, df


X_all, input_df = build_all_features(raw_df)
print('Full feature matrix:', X_all.shape)
display(X_all.head())


## 6. Prediction

Selected features are read from the released `.txt` files. No feature indices are hardcoded here.


In [ ]:
def select_features_by_name(X_all: pd.DataFrame, selected_features):
    missing = [c for c in selected_features if c not in X_all.columns]
    if missing:
        raise ValueError(f'Selected features missing from X_all: {missing[:20]}')
    return X_all.loc[:, selected_features]


X_mobility = select_features_by_name(X_all, mobility_selected_features)
X_carrier = select_features_by_name(X_all, carrier_selected_features)

mobility_pred = mobility_model.predict(X_mobility)
carrier_pred = carrier_model.predict(X_carrier)

mobility_proba = mobility_model.predict_proba(X_mobility) if hasattr(mobility_model, 'predict_proba') else None
carrier_proba = carrier_model.predict_proba(X_carrier) if hasattr(carrier_model, 'predict_proba') else None

results = pd.DataFrame({
    'entry_id': input_df['entry_id'] if 'entry_id' in input_df.columns else [f'ENTRY_{i+1:03d}' for i in range(len(input_df))],
    'polymer_name': input_df['polymer_name'] if 'polymer_name' in input_df.columns else '',
    'pred_mobility_label': mobility_pred,
    'pred_carrier_type': carrier_pred,
})

if mobility_proba is not None:
    for idx, cls in enumerate(mobility_model.classes_):
        results[f'mobility_probability_{cls}'] = mobility_proba[:, idx]

if carrier_proba is not None:
    for idx, cls in enumerate(carrier_model.classes_):
        safe_cls = str(cls).replace(' ', '_').replace('/', '_')
        results[f'carrier_probability_{safe_cls}'] = carrier_proba[:, idx]

display(results)


## 7. Save Prediction Results


In [ ]:
results.to_csv(PREDICTION_OUTPUT_CSV, index=False, encoding='utf-8')
PREDICTION_OUTPUT_JSON.write_text(
    json.dumps(results.where(pd.notna(results), None).to_dict(orient='records'), indent=2, ensure_ascii=False),
    encoding='utf-8'
)

print('Saved CSV:', PREDICTION_OUTPUT_CSV)
print('Saved JSON:', PREDICTION_OUTPUT_JSON)
